# 02 — Demographics and outcomes

Now that I know what's in the data (notebook 01), I want to see who's in it and what happens to them. The plan: build a single ICU-stay-level table that joins patient demographics with admission outcomes, then look at the distributions and how mortality breaks down by subgroup.

The N is small (~100 patients), so the subgroup numbers here are illustrative — but they're the same view I'll need at scale, and the code will carry over to the full MIMIC-IV.

## Setup

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data")
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"

con = duckdb.connect()
sns.set_theme(style="whitegrid", context="notebook")

## Building a stay-level table

Each row = one ICU stay, joined with patient demographics from `patients` and admission-level info (including the mortality outcome) from `admissions`.

In [ ]:
query = f"""
SELECT
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    i.intime,
    i.outtime,
    i.los,
    i.first_careunit,
    p.gender,
    p.anchor_age,
    a.race,
    a.insurance,
    a.marital_status,
    a.hospital_expire_flag
FROM '{ICU / 'icustays.csv.gz'}' AS i
LEFT JOIN '{HOSP / 'patients.csv.gz'}' AS p
    ON i.subject_id = p.subject_id
LEFT JOIN '{HOSP / 'admissions.csv.gz'}' AS a
    ON i.hadm_id = a.hadm_id
"""

stays = con.execute(query).df()
print(f"shape: {stays.shape}")
print(f"unique patients: {stays['subject_id'].nunique()}")
stays.head()

## Demographic distributions

In [ ]:
# Gender
gender_counts = stays["gender"].value_counts(dropna=False)
print(gender_counts)

fig, ax = plt.subplots(figsize=(5, 3))
sns.countplot(data=stays, x="gender", ax=ax, order=gender_counts.index)
ax.set_title("Gender distribution (ICU stays)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Age (anchor_age)
print(stays["anchor_age"].describe())

fig, ax = plt.subplots(figsize=(6, 3))
sns.histplot(data=stays, x="anchor_age", bins=15, ax=ax)
ax.set_title("Anchor age distribution")
ax.set_xlabel("Anchor age (years)")
plt.tight_layout()
plt.show()

In [ ]:
# Race
race_counts = stays["race"].value_counts(dropna=False)
print(race_counts)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=stays, y="race", ax=ax, order=race_counts.index)
ax.set_title("Race distribution (ICU stays)")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

The `race` field in MIMIC is self-reported and split into many sub-categories ("WHITE", "WHITE - OTHER EUROPEAN", "BLACK/AFRICAN AMERICAN", etc.), with overlaps and inconsistencies. I keep it as-is for now, but it deserves caution: race as a biological proxy is exactly the kind of category that the equity-in-clinical-AI literature critiques. For the fairness audit I'll use it as a grouping variable while being explicit about what that does and doesn't measure.

In [ ]:
# Insurance
insurance_counts = stays["insurance"].value_counts(dropna=False)
print(insurance_counts)

fig, ax = plt.subplots(figsize=(6, 3))
sns.countplot(data=stays, x="insurance", ax=ax, order=insurance_counts.index)
ax.set_title("Insurance distribution (ICU stays)")
ax.set_ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Outcome: in-hospital mortality

In [ ]:
mortality_overall = stays["hospital_expire_flag"].mean()
print(f"overall in-hospital mortality rate: {mortality_overall:.1%}")
print()
print(stays["hospital_expire_flag"].value_counts())

## Mortality stratified by subgroup

These are the subgroup rates I'll later compare with model performance. With ~100 stays the numbers will jump around a lot — read them as direction-of-effect, not as point estimates.

In [ ]:
def mortality_by(group):
    return (
        stays.groupby(group, dropna=False)
        .agg(n=("hospital_expire_flag", "size"),
             deaths=("hospital_expire_flag", "sum"),
             rate=("hospital_expire_flag", "mean"))
        .sort_values("n", ascending=False)
    )

mortality_by("gender")

In [ ]:
# Age bands
age_bins = [0, 40, 60, 75, 120]
age_labels = ["<40", "40-59", "60-74", "75+"]
stays["age_band"] = pd.cut(stays["anchor_age"], bins=age_bins, labels=age_labels, right=False)

mortality_by("age_band")

In [ ]:
mortality_by("race")

In [ ]:
mortality_by("insurance")

## Table 1: stratified by outcome

A standard "Table 1" in clinical research compares the cohort across outcome groups. Useful as a unified view of what differs between survivors and non-survivors.

In [ ]:
from tableone import TableOne

columns = ["gender", "anchor_age", "race", "insurance", "los", "first_careunit"]
categorical = ["gender", "race", "insurance", "first_careunit"]
groupby = "hospital_expire_flag"

table1 = TableOne(
    stays,
    columns=columns,
    categorical=categorical,
    groupby=groupby,
    pval=True,
)
print(table1.tabulate(tablefmt="grid"))

## Takeaways

- I have one ICU-stay-level table with demographics + outcome, ready to be filtered into an analytic cohort.
- The N is small enough that some subgroups (specific race categories, age extremes) will have very few patients. p-values here are mostly noise.
- The `race` field will need a careful handling decision in the fairness audit — likely collapsing small categories into broader groupings, or reporting both ways.
- The mortality rate looks reasonable for ICU data; the per-subgroup rates show direction but nothing more at this sample size.

Next (notebook 03): define a clean analytic cohort — first ICU stay per patient, adults, minimum length-of-stay filter — and export it for the feature extraction step.